In [ ]:
# Benjamin_ExtractGraves_Jewish.py
# 4/27/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(2, 4)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract Benjamin Graves Jewish Extract '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileIn="Benjamin_Cemeteries_Jewish_Extract"
fileOutput="Benjamin_Graves_Jewish_Extract"

if append==1:fileIn='Benjamin_Cemeteries_Jewish_X'

fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\Graves\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Graves\\"+fileOutput+"_Interim.csv"

cemetery_frame_df_header=['cemetery_id','memorial_count','cemetery','city','county','state','state_id','county_id']

cemetery_info_df=pd.read_csv(fileCemeteriesIn)

cemetery_info_df.drop(cemetery_info_df[pd.to_numeric(cemetery_info_df["memorial_count"], errors='coerce')<10].index, inplace=True) 

cemetery_id_list=cemetery_info_df["cemetery_id"]
memorial_count_list=cemetery_info_df["memorial_count"]
cemetery_list=cemetery_info_df["cemetery"]
city_list=cemetery_info_df["city"]
county_list=cemetery_info_df["county"]
state_list=cemetery_info_df["state"]
county_id_list=cemetery_info_df["county_id"]
state_id_list=cemetery_info_df["state_id"]

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=1*60

url0 = "https://www.findagrave.com/cemetery/"
url1 ="/memorial-search?ajax=true&skip="
url2 ="&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear=&deathyearfilter=&locationId=&location=&memorialid=&datefilter=&orderby=b&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=&linkedToName=&fuzzyNames=&mcid="

# total_cemeteries=get_total_cemeteries(url0)
date_df=pd.DataFrame([])
cemetery_count=-1
total_frame_count=0
total_cemetery_count=len(cemetery_id_list)
for cemetery_id in cemetery_id_list:
    memorial_count_list
    cemetery_count+=1
    memorial_count=int(memorial_count_list.iloc[cemetery_count])
    cemetery_name=cemetery_list.iloc[cemetery_count]
    cemetery_city=city_list.iloc[cemetery_count]
    cemetery_county=county_list.iloc[cemetery_count]
    cemetery_state=state_list.iloc[cemetery_count]

    frame_count=0
    grave_frame_count=0
    for frame in range(0,memorial_count+1,100):
        total_frame_count+=1
        frame_count+=1
        url=url0+str(cemetery_id)+url1+str(frame)+url2

        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 

        if status_code!=200:print('\tstatus_code:',status_code, 'frame count:',frame_count);break
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()

        capture_json=json.loads(specific_tag_text) 
        grave_count=capture_json['total']
        collection_df=pd.json_normalize(capture_json['collection'])
        try:
            collection_df['maidenName']   
        except:
            collection_df['maidenName']=np.nan           
        try:
            collection_df['birthCountryName']
        except:
            collection_df['birthCountryName']=np.nan   
        try:
            collection_df['Parents']
        except:
            collection_df['Parents']=np.nan    
        try:
            collection_df['lastName']
        except:
            collection_df['lastName']=np.nan   
        try:
            collection_df['firstName']
        except:
            collection_df['firstName']=np.nan               
        try:
            collection_df['Siblings']
        except:
            collection_df['Siblings']=np.nan              
        try:
            collection_df['birthDate']=collection_df['birthDate']
        except:
            collection_df['birthDate']='99-999-99'   
        try:
            collection_df['deathDate'] =collection_df['deathDate']            
        except:
            collection_df['deathDate']='99-999-99'
        try:
            collection_df['birthYear']=collection_df['birthYear'].fillna(9999).astype(int)
        except:
            collection_df['birthYear']=9999   
        try:
            collection_df['birthMonth'] =collection_df['birthMonth'].fillna(99).astype(int) 
        except:
            collection_df['birthMonth']=99           
        try:
            collection_df['deathYear']=collection_df['deathYear'].fillna(9999).astype(int)
        except:
            collection_df['deathYear']=9999  
        try:
            collection_df['deathMonth']=collection_df['deathMonth'].fillna(99).astype(int) 
        except:
            collection_df['deathMonth']=99   

        collection_df['memorial_count']=memorial_count        
        try:
            collection_df['cemeteryId']
        except:
             break
        
        grave_frame_df=pd.concat([collection_df['cemeteryId'],collection_df['cemeteryName'],collection_df['memorial_count'],collection_df['fullName'],collection_df['lastName'], collection_df['firstName'],\
                                      collection_df['maidenName'],collection_df['birthCountryName'],collection_df['birthYear'],collection_df['birthMonth'],collection_df['deathYear'],collection_df['deathMonth'],\
                                      collection_df['birthDate'],collection_df['deathDate'],collection_df['Parents'],\
                                      collection_df['Siblings']], axis=1)            
        graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
        grave_frame_count+=len(grave_frame_df)
        last_death_year=graves_df['deathYear'][graves_df['deathYear'].astype(int)!=9999].max()
        print(f"    --- {process_name:<20}• {cemetery_name:<20}• {cemetery_city:<20}• {cemetery_county:<20}• {cemetery_state:<10}• grave count: {grave_frame_count}/{memorial_count} • Last Death Year: {last_death_year}")
        # print(graves_df[['Parents','Siblings']]) 
        # input('graves_df')
        sleep_time=random_sleep(frame_count)       
        time.sleep(sleep_time)

    # print(graves_df[['Parents','Siblings']]) 
    
    graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    end_time=datetime.datetime.now()
    run_processing_time=end_time-event_time
    event_time=end_time
    run_processing_time_seconds=int(run_processing_time.total_seconds())
    processing_time=end_time-start_time
    processing_time_minutes=int(processing_time.total_seconds())/60        
    print(f"*** {process_name:<20}• {cemetery_name:<20}• {cemetery_city:<20}• {cemetery_state:<10}• grave count: {grave_frame_count} •"+\
          f" cemetery count: {cemetery_count+1}/{total_cemetery_count} • cemetery id: {cemetery_id}• Processing Time: {run_processing_time_seconds} Sec • Cum Processing Time {processing_time_minutes:.1f} min\n")
    # print(graves_df) 
    # input('graves_df')

field_list=['cemeteryId','cemeteryName','memorial_count','fullName','lastName','firstName','maidenName','birthCountryName',\
            'birthYear','birthMonth','deathYear','deathMonth','birthDate','deathDate','Parents','Siblings']
for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\*", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
for item in delete_list:
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


In [156]:
collection_df=pd.json_normalize(capture_json['collection'])
field_names=['lastName','firstName','fullName','birthYear','deathYear','birthMonth','deathMonth','birthDate','deathDate',\
    'cemeteryName','cemeteryId','cityName','stateName','maidenName','Parents','Children', 'Siblings','Spouses']
collection_df[field_names]

,lastName,firstName,fullName,birthYear,deathYear,birthMonth,deathMonth,birthDate,deathDate,cemeteryName,cemeteryId,cityName,stateName,maidenName,Parents,Children,Siblings,Spouses
0,Phillips,Carrie,Carrie M. Phillips,1853,1865,6.0,1.0,3 Jun 1853,11 Jan 1865,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,NaN,NaN,NaN,NaN
1,Batten,Winfield,Winfield Scott Batten,1853,1915,7.0,11.0,9 Jul 1853,4 Nov 1915,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,[Susan Freeland Thornton],NaN,"[Charlotte Lovina Batten, George W. Batten, Lo...",NaN
2,Post,Milla,Milla M. Post,1853,1854,12.0,11.0,11 Dec 1853,13 Nov 1854,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,[Nancy Ann Post],NaN,NaN,NaN
3,Leas,Mary,Mary M. Leas,1854,1855,1.0,8.0,30 Jan 1854,13 Aug 1855,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,NaN,NaN,NaN,NaN
4,Noggle,Helen,Helen E. Noggle,1854,1854,3.0,4.0,4 Mar 1854,5 Apr 1854,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,"[Nancy Helen Wilson Noggle, Perry Noggle]",NaN,"[Almira Lodema Noggle Plank, Charles E. Noggle...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Welbaum,Isabelle,Isabelle Charlotte Welbaum,1918,1918,4.0,5.0,8 Apr 1918,6 May 1918,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,"[Henrietta Sabina Phillippi Welbaum, Ora Micha...",NaN,"[Nona Mae Welbaum Sebasty, Ora Michael Welbaum...",NaN
96,Barber,Kenneth,Kenneth Barber,1921,1927,8.0,3.0,24 Aug 1921,29 Mar 1927,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,"[Effie Ethel Mae Little Bushey, Theodore Barber]",NaN,"[Albert Gilbert Barber, Crystal Barber McClara...",NaN
97,Wright,James,James Wright,1921,2001,NaN,NaN,1921,2001,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,NaN,NaN,NaN,[Allie Mae Slaughter Wright]
98,Holycross,Frederick,Frederick James Fred Holycross,1928,2011,12.0,11.0,16 Dec 1928,17 Nov 2011,Mount Zion Cemetery,1973290,Dayton,Michigan,NaN,[Albert William Holycross],NaN,NaN,[Zella Geraldine Holycross]


In [ ]:


field_list=['cemeteryId','cemeteryName','memorial_count','fullName','lastName','firstName','maidenName','birthCountryName',\
            'birthYear','birthMonth','deathYear','deathMonth','birthDate','deathDate','Parents','Siblings']
for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\*", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
for item in delete_list:
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()

graves_temp_df['Siblings'].loc[37:50]     

In [ ]:
list(collection_df)

In [ ]:
grave_frame_df